# Data Wrangling & EDA — NITIP Project
## End-to-End Pipeline: Raw BPS Data → Clean Processed CSV

**Tujuan:** Memproses seluruh data mentah (CPI, CPI Sektoral, Mortalitas, Gaji, Investasi) menjadi dataset bersih yang siap digunakan untuk model kalkulator pensiun aktuarial.

**Sumber Data:**
- BPS (Badan Pusat Statistik) Indonesia
- BPJS Ketenagakerjaan (Tabel Mortalitas)
- Yahoo Finance / Investing.com (IHSG, Obligasi)

In [ ]:
import re, warnings, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.stats import linregress
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.figsize':(14,5),'font.size':11,'axes.grid':True,'grid.alpha':0.3})

ROOT = Path('.').resolve()
if ROOT.name in ('scripts', 'notebooks'): ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)
print(f"Root: {ROOT}")
print(f"Raw:  {RAW}")

## Helper Functions
Fungsi-fungsi utilitas untuk parsing angka format Indonesia dan BPS.

In [ ]:
def parse_num(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s in ('-','','nan','#N/A'): return np.nan
    s = re.sub(r'[^\d.,-]','',s)
    if not s: return np.nan
    has_c, has_d = ',' in s, '.' in s
    if has_c and has_d:
        if s.rfind(',') > s.rfind('.'): s = s.replace('.','').replace(',','.')
        else: s = s.replace(',','')
    elif has_c: s = s.replace(',','.')
    elif has_d:
        parts = s.split('.')
        if len(parts)>1 and all(len(p)==3 for p in parts[1:]): s = s.replace('.','')
    try: return float(s)
    except: return np.nan

def parse_dot(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s in ('-','','nan'): return np.nan
    s = re.sub(r'[^\d.-]','',s)
    try: return float(s)
    except: return np.nan

def find_indonesia_row(fp, col_idx=0):
    df = pd.read_csv(fp, header=0, dtype=str, encoding='utf-8-sig')
    col = df.columns[col_idx]
    mask = df[col].str.strip().str.upper() == 'INDONESIA'
    if not mask.any(): mask = df[col].str.strip().str.upper().str.contains('INDONESIA',na=False)
    return df[mask].iloc[0]

def monthly_from_bps(series):
    records = []
    for col, val in series.items():
        col = str(col).strip()
        if re.match(r'^\d{2}/\d{4}$', col):
            mm, yyyy = col.split('/')
            records.append({'year_month':f'{yyyy}-{mm}','index_value':parse_num(val)})
    if not records: return pd.DataFrame(columns=['year_month','index_value'])
    df = pd.DataFrame(records)
    df['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m')
    return df.sort_values('year_month').reset_index(drop=True)

print("Helper functions loaded.")

---
## 1. CPI Umum (Inflasi) — Chain-Linking 3 Base Year

### Masalah
BPS mengubah tahun dasar CPI 3 kali:
- **2015–2019**: base 2012=100
- **2020–2023**: base 2018=100  
- **2024–2026**: base 2022=100

Jika digabung mentah, indeks melompat dari ~139 ke ~104 di titik transisi, menyebabkan inflasi palsu -25%.

### Solusi: Splice Chain-Linking
Kalikan setiap seri dengan faktor skala agar level indeksnya menyambung mulus.

In [ ]:
CPI_DIR = RAW / 'CPI'
def load_cpi(pattern):
    matches = [f for f in CPI_DIR.glob('*.csv') if pattern.lower() in f.name.lower()]
    return monthly_from_bps(find_indonesia_row(matches[0]))

df_s1 = load_cpi('2015-2019')  # base 2012=100
df_s2 = load_cpi('2020-2023')  # base 2018=100
df_s3 = load_cpi('2024-2026')  # base 2022=100
df_s3 = df_s3[df_s3['year_month'] <= '2025-12-01']

print(f"Seri 1 (2012=100): {len(df_s1)} bulan, {df_s1.year_month.min().strftime('%Y-%m')} s.d. {df_s1.year_month.max().strftime('%Y-%m')}")
print(f"Seri 2 (2018=100): {len(df_s2)} bulan, {df_s2.year_month.min().strftime('%Y-%m')} s.d. {df_s2.year_month.max().strftime('%Y-%m')}")
print(f"Seri 3 (2022=100): {len(df_s3)} bulan, {df_s3.year_month.min().strftime('%Y-%m')} s.d. {df_s3.year_month.max().strftime('%Y-%m')}")

### Visualisasi: 3 Seri SEBELUM Chain-Linking
Perhatikan gap/lompatan besar di titik transisi.

In [ ]:
fig, ax = plt.subplots(figsize=(14,6))
ax.plot(df_s1.year_month, df_s1.index_value, 'b-o', ms=3, label='Seri 1 (2012=100)')
ax.plot(df_s2.year_month, df_s2.index_value, 'r-o', ms=3, label='Seri 2 (2018=100)')
ax.plot(df_s3.year_month, df_s3.index_value, 'g-o', ms=3, label='Seri 3 (2022=100)')
ax.axvline(pd.Timestamp('2020-01-01'), color='gray', ls='--', alpha=0.7, label='Transisi base year')
ax.axvline(pd.Timestamp('2024-01-01'), color='gray', ls='--', alpha=0.7)
ax.set_title('CPI Umum — 3 Seri dengan Base Year Berbeda (SEBELUM Chain-Linking)', fontsize=14, fontweight='bold')
ax.set_ylabel('Indeks Harga Konsumen')
ax.legend()
plt.tight_layout()
plt.show()
print("\nTerlihat jelas: indeks TURUN drastis di titik transisi. Ini BUKAN deflasi, ini perbedaan base year!")

### Proses Chain-Linking

**Langkah 1:** Hitung scale factor di titik sambung
```
scale_2 = Indeks_Des2019_base2012 / Indeks_Jan2020_base2018
scale_3 = (Indeks_Des2023_base2018 × scale_2) / Indeks_Jan2024_base2022
```

In [ ]:
def get_val(df, ym):
    mask = df['year_month'].dt.strftime('%Y-%m') == ym
    v = df.loc[mask,'index_value'].values
    return v[0] if len(v) and not np.isnan(v[0]) else None

v_dec19 = get_val(df_s1, '2019-12')
v_jan20 = get_val(df_s2, '2020-01')
scale_2 = v_dec19 / v_jan20

v_dec23 = get_val(df_s2, '2023-12')
v_dec23_scaled = v_dec23 * scale_2
v_jan24 = get_val(df_s3, '2024-01')
scale_3 = v_dec23_scaled / v_jan24

print(f"Des 2019 (base 2012=100) = {v_dec19}")
print(f"Jan 2020 (base 2018=100) = {v_jan20}")
print(f"scale_2 = {v_dec19} / {v_jan20} = {scale_2:.4f}")
print()
print(f"Des 2023 (base 2018=100) = {v_dec23}")
print(f"Des 2023 setelah scaling = {v_dec23} x {scale_2:.4f} = {v_dec23_scaled:.2f}")
print(f"Jan 2024 (base 2022=100) = {v_jan24}")
print(f"scale_3 = {v_dec23_scaled:.2f} / {v_jan24} = {scale_3:.4f}")

In [ ]:
# Terapkan scaling
df_s1['base_year'] = '2012=100'
df_s2['base_year'] = '2018=100'
df_s3['base_year'] = '2022=100'

df_a = df_s1.copy(); df_a['idx'] = df_a['index_value']  # sudah base 2012
df_b = df_s2.copy(); df_b['idx'] = df_b['index_value'] * scale_2
df_c = df_s3.copy(); df_c['idx'] = df_c['index_value'] * scale_3

# Trim overlap
df_a = df_a[df_a.year_month < '2020-01-01']
df_b = df_b[(df_b.year_month >= '2020-01-01') & (df_b.year_month < '2024-01-01')]
df_c = df_c[df_c.year_month >= '2024-01-01']

cpi = pd.concat([df_a[['year_month','idx','base_year']], df_b[['year_month','idx','base_year']], df_c[['year_month','idx','base_year']]]).sort_values('year_month').reset_index(drop=True)
cpi['idx'] = cpi['idx'].interpolate('linear').ffill().bfill()
cpi['mom'] = cpi['idx'].pct_change(1)*100
cpi['yoy'] = cpi['idx'].pct_change(12)*100

# Validasi transisi
v1 = cpi.loc[cpi.year_month.dt.strftime('%Y-%m')=='2019-12','idx'].values[0]
v2 = cpi.loc[cpi.year_month.dt.strftime('%Y-%m')=='2020-01','idx'].values[0]
print(f"VALIDASI TRANSISI Des2019->Jan2020: {(v2/v1-1)*100:+.4f}% MoM")
v3 = cpi.loc[cpi.year_month.dt.strftime('%Y-%m')=='2023-12','idx'].values[0]
v4 = cpi.loc[cpi.year_month.dt.strftime('%Y-%m')=='2024-01','idx'].values[0]
print(f"VALIDASI TRANSISI Des2023->Jan2024: {(v4/v3-1)*100:+.4f}% MoM")
print("(Harus mendekati 0% — artinya sambungan mulus)")

### Visualisasi: SETELAH Chain-Linking

In [ ]:
fig, axes = plt.subplots(2,1,figsize=(14,10))
ax1 = axes[0]
colors = {'2012=100':'#2196F3','2018=100':'#FF5722','2022=100':'#4CAF50'}
for by, grp in cpi.groupby('base_year'):
    ax1.plot(grp.year_month, grp.idx, '-', color=colors.get(by,'gray'), lw=2, label=f'Base {by}')
ax1.axvline(pd.Timestamp('2020-01-01'),color='gray',ls='--',alpha=0.5)
ax1.axvline(pd.Timestamp('2024-01-01'),color='gray',ls='--',alpha=0.5)
ax1.set_title('CPI Umum — SETELAH Chain-Linking (Rebased ke 2012=100)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Indeks (rebased)')
ax1.legend()

ax2 = axes[1]
valid = cpi.dropna(subset=['yoy'])
ax2.plot(valid.year_month, valid.yoy, 'b-', lw=1.5)
ax2.axhline(0, color='red', ls='--', alpha=0.5)
ax2.fill_between(valid.year_month, valid.yoy, 0, alpha=0.15)
ax2.set_title('Inflasi Year-over-Year (%)', fontsize=14, fontweight='bold')
ax2.set_ylabel('YoY (%)')
ax2.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f%%'))
plt.tight_layout()
plt.show()
print("Grafik atas: indeks menyambung mulus tanpa lompatan.")
print("Grafik bawah: inflasi YoY konsisten 1-5%, sesuai data BPS resmi.")

In [ ]:
# Simpan
cpi['ym_str'] = cpi.year_month.dt.strftime('%Y-%m')
cpi.rename(columns={'idx':'cpi_index_rebased','ym_str':'year_month_out'})[['year_month_out','cpi_index_rebased','mom','yoy','base_year']].rename(columns={'year_month_out':'year_month','mom':'inflation_mom_pct','yoy':'inflation_yoy_pct'}).to_csv(PROC/'cpi_monthly.csv', index=False)

cpi_dec = cpi[cpi.year_month.dt.month==12].copy()
cpi_dec['year'] = cpi_dec.year_month.dt.year
cpi_dec['inflasi_pct'] = cpi_dec.yoy.round(2)
cpi_dec[['year','inflasi_pct','base_year']].to_csv(PROC/'cpi_clean.csv', index=False)
print("Saved: cpi_monthly.csv, cpi_clean.csv")
cpi_dec[['year','inflasi_pct','base_year']]